# Airspace Object Detection, Week 1: Baseline

This notebook downloads AOD 4 from Roboflow, trains a YOLO baseline, and saves per-class results and a confusion matrix.

**Before running:**
1. In the right sidebar, under **Session options**, set **Accelerator** to **GPU T4 x2** and turn **Internet** on. Internet needs a phone-verified Kaggle account.
2. Go to **Add-ons → Secrets** and add a secret named `ROBOFLOW_API_KEY` set to your **new** Roboflow key. The key then never appears in the notebook.
3. Click **Run All**. Training takes roughly 1 to 2 hours on a T4.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
# 1. Download AOD 4 in YOLO format (key comes from Kaggle Secrets)
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)
project = rf.workspace("airborne-object-detection").project("airborne-object-detection-4-aod4")
dataset = project.version(6).download("yolov8", location="/kaggle/working/aod4")
print(dataset.location)

In [ ]:
# 2. Look at the data: class names and image counts per split
import yaml, os, glob
data_yaml = os.path.join(dataset.location, "data.yaml")
cfg = yaml.safe_load(open(data_yaml))
print("Classes:", cfg["names"])
for split in ["train", "valid", "test"]:
    n = len(glob.glob(os.path.join(dataset.location, split, "images", "*")))
    print(f"{split}: {n} images")

In [ ]:
# 3. Count boxes per class in the training set (checks for class imbalance)
from collections import Counter
counts = Counter()
for f in glob.glob(os.path.join(dataset.location, "train", "labels", "*.txt")):
    for line in open(f):
        if line.strip():
            counts[int(line.split()[0])] += 1
for i, name in enumerate(cfg["names"]):
    print(f"{name:12s} {counts[i]}")

In [ ]:
# 4. Train the baseline (640 px). Two T4 GPUs: device=[0,1]
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=32,
    device=[0, 1],
    project="/kaggle/working/runs",
    name="aod4_baseline_640",
    patience=15,
)

In [ ]:
# 5. Evaluate on the test split: overall mAP and AP per class
best = YOLO("/kaggle/working/runs/aod4_baseline_640/weights/best.pt")
metrics = best.val(data=data_yaml, split="test", imgsz=640, plots=True,
                   project="/kaggle/working/runs", name="aod4_test")
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
for i, name in enumerate(cfg["names"]):
    print(f"{name:12s} AP50-95 = {metrics.box.maps[i]:.3f}")

In [ ]:
# 6. Show the confusion matrix (which classes get mixed up)
from IPython.display import Image, display
display(Image("/kaggle/working/runs/aod4_test/confusion_matrix_normalized.png", width=700))

In [ ]:
# 7. Look at predictions on a few test images
test_imgs = sorted(glob.glob(os.path.join(dataset.location, "test", "images", "*")))[:8]
results = best.predict(test_imgs, imgsz=640, conf=0.25, save=True,
                       project="/kaggle/working/runs", name="aod4_samples")
for p in sorted(glob.glob("/kaggle/working/runs/aod4_samples/*"))[:8]:
    display(Image(p, width=500))

## Write down before moving on

- Overall mAP50-95 and each class's AP. This is the baseline row of your results table.
- The two classes most often confused, from the confusion matrix.
- 2 or 3 test images where the model was wrong, and why you think it failed. Start your failure gallery with these.

**Next (week 2):** retrain at `imgsz=1280` and compare AP for small objects, then write your own PyTorch evaluation script.